# MINI Cells — Experiment 003: Native Trainability Bridge

This notebook isolates the gap between the already-learnable floating-point Echo architecture
and the deterministic Q8.8 / SIGN-SPSA training path intended for MiniJAM/PVM.

It answers four questions:

1. Does the 4,476-parameter Echo architecture still learn under Q8.8-aware forward dynamics?
2. Does the production-style margin objective remain trainable with gradients?
3. How much trainability is lost when replacing Adam with deterministic two-sided SIGN-SPSA?
4. Can deterministic blockwise SPSA recover useful learning while preserving a PLUS/MINUS update shape?

The notebook **does not modify the production runtime**. It produces research evidence and candidate
parameters for a later protocol/runtime decision.

### Decision gates

- **Q8.8 gate:** QAT token accuracy should approach the FP32 baseline.
- **Objective gate:** Q8.8 + margin should remain strongly trainable with gradients.
- **Native optimizer gate:** a deterministic SPSA variant should materially outperform the current
  global 4,476-dimensional update from the same genesis.
- **Kernel bridge:** an exported Q8.8 checkpoint is sampled through `minicells.fixed_v1.predict`
  to check agreement with the production-style integer inference mirror.

In [ ]:
import json, math, os, platform, random, re, sys, time, hashlib, struct
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "research").exists():
    ROOT = Path("/kaggle/working/mini-cells")
if not (ROOT / "research").exists():
    raise RuntimeError("Run this notebook from a mini-cells checkout (or /kaggle/working/mini-cells).")

sys.path.insert(0, str(ROOT / "research"))

from minicells.config import load_config, resolved_config
from minicells.data import CopyDataGenerator, fixed_dataset
from minicells.evaluate import evaluate
from minicells.fixed_v1 import pack_model, predict as fixed_predict
from minicells.metrics import masked_cross_entropy
from minicells.model import EchoModel
from minicells.ops import architecture_stats
from minicells.reproducibility import set_global_seed
from minicells.vocab import CharVocab

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = ROOT / "results" / "trainability-v1"
OUT.mkdir(parents=True, exist_ok=True)

FULL_RUN = True
SEED = 3

if FULL_RUN:
    ADAM_STEPS = 1500
    SPSA_GENERATIONS = 1800
    VAL_EXAMPLES = 2048
    EVAL_EVERY = 100
else:
    ADAM_STEPS = 250
    SPSA_GENERATIONS = 250
    VAL_EXAMPLES = 512
    EVAL_EVERY = 50

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "full_run": FULL_RUN,
    "adam_steps": ADAM_STEPS,
    "spsa_generations": SPSA_GENERATIONS,
})

In [ ]:
vocab = CharVocab()
base_config = resolved_config(load_config(ROOT / "configs" / "echo-v0.yaml"), len(vocab))
model_kwargs = {k: v for k, v in base_config["model"].items() if k != "vocab_size"}
model_kwargs["vocab_size"] = len(vocab)

data_args = dict(
    min_length=base_config["data"]["min_length"],
    max_length=base_config["data"]["max_length"],
    num_cells=base_config["model"]["num_cells"],
    random_fraction=base_config["data"]["random_fraction"],
)

validation = fixed_dataset(
    vocab,
    seed=10003,
    examples=VAL_EXAMPLES,
    **data_args,
).to(DEVICE)

probe_validation = fixed_dataset(
    vocab,
    seed=19003,
    examples=min(512, VAL_EXAMPLES),
    **data_args,
).to(DEVICE)

reference = EchoModel(**model_kwargs)
stats = architecture_stats(reference)
assert stats["parameter_count"] == 4476, stats
print(stats)

## Q8.8-aware differentiable model

The production kernel stores each parameter as a signed Q8.8 integer and rounds linear outputs back
to Q8.8. Hidden state is clamped to `[-1, 1]`. For gradient experiments we use a straight-through
estimator (STE): the forward pass sees quantized values while gradients pass through the rounding
operation.

This is deliberately a **bridge model**, not a claim of bit-for-bit PVM arithmetic. Exact integer
sampling is checked later with `fixed_v1`.

In [ ]:
Q_SCALE = 256.0
Q_MIN = -2048
Q_MAX = 2048

def symmetric_round_ste(x: torch.Tensor) -> torch.Tensor:
    q = torch.sign(x) * torch.floor(torch.abs(x) * Q_SCALE + 0.5) / Q_SCALE
    return x + (q - x).detach()

def qparam_ste(x: torch.Tensor) -> torch.Tensor:
    clipped = torch.clamp(x, Q_MIN / Q_SCALE, Q_MAX / Q_SCALE)
    q = torch.sign(clipped) * torch.floor(torch.abs(clipped) * Q_SCALE + 0.5) / Q_SCALE
    return x + (q - x).detach()

def qstate_ste(x: torch.Tensor) -> torch.Tensor:
    clipped = torch.clamp(x, -1.0, 1.0)
    q = torch.sign(clipped) * torch.floor(torch.abs(clipped) * Q_SCALE + 0.5) / Q_SCALE
    return x + (q - x).detach()

class QATEchoModel(EchoModel):
    def _qlinear(self, x, layer):
        w = qparam_ste(layer.weight)
        b = qparam_ste(layer.bias) if layer.bias is not None else None
        # Production rounds the accumulator back to Q8.8 but does not clamp
        # the linear output to parameter bounds.
        return symmetric_round_ste(F.linear(x, w, b))

    def forward(self, input_ids: torch.Tensor, return_state: bool = False):
        if input_ids.ndim != 2 or input_ids.shape[1] != self.num_cells:
            raise ValueError(f"input_ids must have shape [batch, {self.num_cells}]")
        embedded = F.embedding(
            input_ids,
            qparam_ste(self.embedding.weight),
            padding_idx=self.embedding.padding_idx,
        )
        state = self.initial_state(input_ids)
        for _ in range(self.iterations):
            update_input = torch.cat((self._neighborhood(state), embedded), dim=-1)
            hidden = F.relu(self._qlinear(update_input, self.update_in))
            delta = self._qlinear(hidden, self.update_out)
            state = qstate_ste(state + self.residual_scale * delta)
        logits = self._qlinear(state, self.output)
        return (logits, state) if return_state else logits

def masked_margin_loss(logits, targets, mask, margin=1.0, reduction="mean"):
    target_logits = logits.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    masked = logits.clone()
    masked.scatter_(-1, targets.unsqueeze(-1), float("-inf"))
    other = masked.max(dim=-1).values
    loss = F.relu(margin - (target_logits - other))
    values = loss[mask]
    if reduction == "sum":
        return values.sum()
    return values.mean()

def quantize_parameter_tensor(x: torch.Tensor) -> torch.Tensor:
    scaled = torch.sign(x) * torch.floor(torch.abs(x) * Q_SCALE + 0.5)
    return torch.clamp(scaled, Q_MIN, Q_MAX).to(torch.int32)

def flat_q(model: nn.Module) -> torch.Tensor:
    chunks = [quantize_parameter_tensor(p.detach().cpu()).reshape(-1) for p in model.parameters()]
    out = torch.cat(chunks)
    assert out.numel() == 4476
    return out

def load_flat_q(model: nn.Module, q: torch.Tensor) -> None:
    cursor = 0
    with torch.no_grad():
        for p in model.parameters():
            n = p.numel()
            p.copy_(q[cursor:cursor+n].reshape(p.shape).to(p.device, torch.float32) / Q_SCALE)
            cursor += n
    assert cursor == 4476

In [ ]:
def train_gradient_variant(name, model, loss_kind, steps=ADAM_STEPS, batch_size=256, lr=1e-3):
    set_global_seed(SEED)
    model = model.to(DEVICE)
    training = CopyDataGenerator(vocab, seed=SEED, **data_args)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    rows = []
    start = time.time()

    initial = evaluate(model, validation)
    rows.append({"step": 0, **initial})

    for step in range(1, steps + 1):
        model.train()
        batch = training.batch(batch_size, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch.input_ids)
        if loss_kind == "ce":
            loss = masked_cross_entropy(logits, batch.target_ids, batch.mask)
        elif loss_kind == "margin":
            loss = masked_margin_loss(logits, batch.target_ids, batch.mask, margin=1.0)
        else:
            raise ValueError(loss_kind)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if isinstance(model, QATEchoModel):
            # Keep shadow parameters inside the exact V1 representable range.
            with torch.no_grad():
                for p in model.parameters():
                    p.clamp_(Q_MIN / Q_SCALE, Q_MAX / Q_SCALE)

        if step % EVAL_EVERY == 0 or step == steps:
            metrics = evaluate(model, validation)
            rows.append({"step": step, **metrics})
            print(
                f"{name:18s} step={step:5d} "
                f"loss={metrics['loss']:.4f} "
                f"token={metrics['token_accuracy']:.4%} "
                f"exact={metrics['exact_sequence_accuracy']:.4%}"
            )

    frame = pd.DataFrame(rows)
    frame.to_csv(OUT / f"{name}.csv", index=False)
    summary = {
        "name": name,
        "loss_kind": loss_kind,
        "steps": steps,
        "seconds": time.time() - start,
        "final_token_accuracy": float(frame.iloc[-1]["token_accuracy"]),
        "final_exact_sequence_accuracy": float(frame.iloc[-1]["exact_sequence_accuracy"]),
        "best_token_accuracy": float(frame["token_accuracy"].max()),
    }
    return model, frame, summary

## Runs A–C — locate the first failure boundary

- **A / fp32-ce:** original architecture, AdamW, cross entropy.
- **B / q88-ce:** Q8.8-aware forward pass, AdamW, cross entropy.
- **C / q88-margin:** Q8.8-aware forward pass, AdamW, production-style margin objective.

If A passes but B collapses, quantization/dynamics are the problem.
If B passes but C collapses, the objective is the problem.
If all three learn, the main bottleneck is the native optimizer.

In [ ]:
fp32_model, fp32_curve, fp32_summary = train_gradient_variant(
    "fp32-ce", EchoModel(**model_kwargs), "ce"
)

qat_ce_model, qat_ce_curve, qat_ce_summary = train_gradient_variant(
    "q88-ce", QATEchoModel(**model_kwargs), "ce"
)

qat_margin_model, qat_margin_curve, qat_margin_summary = train_gradient_variant(
    "q88-margin", QATEchoModel(**model_kwargs), "margin"
)

gradient_summaries = pd.DataFrame([fp32_summary, qat_ce_summary, qat_margin_summary])
gradient_summaries

In [ ]:
plt.figure(figsize=(9, 5))
for name, frame in [
    ("FP32 + CE", fp32_curve),
    ("Q8.8 + CE", qat_ce_curve),
    ("Q8.8 + margin", qat_margin_curve),
]:
    plt.plot(frame["step"], frame["token_accuracy"], label=name)
plt.xlabel("optimizer step")
plt.ylabel("validation token accuracy")
plt.title("Gradient trainability bridge")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "gradient-trainability.png")
plt.show()

## Export a Q8.8 pretrained candidate and probe the fixed integer mirror

This does not change the runtime genesis. It gives us a concrete answer to a separate question:
can a trained floating/QAT checkpoint be represented by the current 8,952-byte model format and
still behave usefully through the fixed integer inference mirror?

In [ ]:
qat_q = flat_q(qat_ce_model)
qat_bytes = pack_model(qat_q.tolist())
assert len(qat_bytes) == 8952
(OUT / "pretrained-q88-model.bin").write_bytes(qat_bytes)

probe_texts = [
    "hello jam",
    "mini cells",
    "local neural",
    "echo",
    "abc123",
    "jam!",
    "small",
    "learn",
]

fixed_samples = []
for text in probe_texts:
    prediction = fixed_predict(qat_bytes, text)
    fixed_samples.append({
        "input": text,
        "prediction": prediction,
        "exact": prediction == text,
        "char_matches": sum(a == b for a, b in zip(text, prediction)),
        "length": len(text),
    })
fixed_samples_df = pd.DataFrame(fixed_samples)
fixed_samples_df

In [ ]:
def model_hash_q(q: torch.Tensor) -> bytes:
    arr = q.detach().cpu().numpy().astype("<i2", copy=False)
    h = hashlib.blake2b(digest_size=32)
    h.update(b"mini-cells:model:v1")
    h.update(arr.tobytes())
    return h.digest()

def production_delta(parent_hash: bytes, generation: int) -> np.ndarray:
    h = hashlib.blake2b(digest_size=32)
    h.update(b"mini-cells:spsa:v1")
    h.update(parent_hash)
    h.update(int(generation).to_bytes(8, "little", signed=False))
    seed = int.from_bytes(h.digest()[:8], "little", signed=False)

    idx = np.arange(1, 4477, dtype=np.uint64)
    with np.errstate(over="ignore"):
        z = np.uint64(seed) + np.uint64(0x9E3779B97F4A7C15) * idx
        z = (z ^ (z >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
        z = (z ^ (z >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
        z = z ^ (z >> np.uint64(31))
    return np.where((z & np.uint64(1)) == 0, -1, 1).astype(np.int32)

def parse_repository_genesis() -> torch.Tensor:
    text = (ROOT / "service" / "generated" / "genesis_model.rs").read_text()
    match = re.search(r"GENESIS_MODEL:\s*\[i16;\s*4476\]\s*=\s*\[(.*?)\];", text, re.S)
    if not match:
        raise RuntimeError("could not parse service/generated/genesis_model.rs")
    values = [int(x) for x in re.findall(r"-?\d+", match.group(1))]
    if len(values) != 4476:
        raise RuntimeError(f"expected 4476 genesis parameters, got {len(values)}")
    return torch.tensor(values, dtype=torch.int32)

GENESIS_Q = parse_repository_genesis()
print({
    "genesis_parameters": GENESIS_Q.numel(),
    "min_q": int(GENESIS_Q.min()),
    "max_q": int(GENESIS_Q.max()),
    "hash": model_hash_q(GENESIS_Q).hex(),
})

## Runs D–E — deterministic two-sided optimizer sweep

Every generation evaluates exactly two candidates, PLUS and MINUS.

`production-global` reproduces the important optimizer geometry of the current production rule:
one ±1 direction spans all 4,476 parameters, `perturbation_q=4`, `step_q=1`, batch size 4.

The blockwise variants keep the same two-sided shape but only perturb a deterministic contiguous
parameter block each generation. They are research candidates, not protocol changes.

In [ ]:
def batch_margin_stats(model, batch):
    model.eval()
    with torch.no_grad():
        logits = model(batch.input_ids)
        loss = masked_margin_loss(
            logits, batch.target_ids, batch.mask, margin=1.0, reduction="sum"
        )
        pred = logits.argmax(-1)
        correct = ((pred == batch.target_ids) & batch.mask).sum().item()
        total = batch.mask.sum().item()
    return float(loss.item()), int(correct), int(total)

def eval_q(model, q, dataset):
    load_flat_q(model, q)
    return evaluate(model, dataset)

def run_two_sided(
    name,
    *,
    generations,
    block_size,
    batch_size,
    perturbation_q=4,
    step_q=1,
    seed=SEED,
):
    q = GENESIS_Q.clone()
    model = QATEchoModel(**model_kwargs).to(DEVICE)
    training = CopyDataGenerator(vocab, seed=seed, **data_args)
    rows = []
    start = time.time()
    nblocks = math.ceil(q.numel() / block_size)

    initial = eval_q(model, q, probe_validation)
    rows.append({
        "generation": 0,
        "token_accuracy": initial["token_accuracy"],
        "exact_sequence_accuracy": initial["exact_sequence_accuracy"],
        "updated": False,
    })

    for generation in range(generations):
        batch = training.batch(batch_size, DEVICE)
        parent_hash = model_hash_q(q)
        direction = production_delta(parent_hash, generation)

        if block_size < q.numel():
            block = generation % nblocks
            lo = block * block_size
            hi = min(q.numel(), lo + block_size)
            mask = np.zeros(q.numel(), dtype=np.int32)
            mask[lo:hi] = 1
            direction *= mask

        d = torch.from_numpy(direction)
        plus_q = torch.clamp(q + perturbation_q * d, Q_MIN, Q_MAX)
        minus_q = torch.clamp(q - perturbation_q * d, Q_MIN, Q_MAX)

        load_flat_q(model, plus_q)
        plus_loss, plus_correct, total = batch_margin_stats(model, batch)

        load_flat_q(model, minus_q)
        minus_loss, minus_correct, _ = batch_margin_stats(model, batch)

        updated = plus_loss != minus_loss
        if plus_loss < minus_loss:
            q = torch.clamp(q + step_q * d, Q_MIN, Q_MAX)
        elif minus_loss < plus_loss:
            q = torch.clamp(q - step_q * d, Q_MIN, Q_MAX)

        completed = generation + 1
        if completed % EVAL_EVERY == 0 or completed == generations:
            metrics = eval_q(model, q, probe_validation)
            rows.append({
                "generation": completed,
                "token_accuracy": metrics["token_accuracy"],
                "exact_sequence_accuracy": metrics["exact_sequence_accuracy"],
                "updated": updated,
                "plus_loss": plus_loss,
                "minus_loss": minus_loss,
                "plus_correct": plus_correct,
                "minus_correct": minus_correct,
                "batch_tokens": total,
            })
            print(
                f"{name:22s} gen={completed:5d} "
                f"token={metrics['token_accuracy']:.4%} "
                f"exact={metrics['exact_sequence_accuracy']:.4%} "
                f"last={'update' if updated else 'tie'}"
            )

    frame = pd.DataFrame(rows)
    frame.to_csv(OUT / f"{name}.csv", index=False)
    summary = {
        "name": name,
        "generations": generations,
        "block_size": block_size,
        "batch_size": batch_size,
        "perturbation_q": perturbation_q,
        "step_q": step_q,
        "seconds": time.time() - start,
        "final_token_accuracy": float(frame.iloc[-1]["token_accuracy"]),
        "best_token_accuracy": float(frame["token_accuracy"].max()),
        "final_exact_sequence_accuracy": float(frame.iloc[-1]["exact_sequence_accuracy"]),
        "final_model_hash": model_hash_q(q).hex(),
    }
    return q, frame, summary

In [ ]:
primary_sweep = [
    dict(name="production-global", block_size=4476, batch_size=4, perturbation_q=4, step_q=1),
    dict(name="block-512", block_size=512, batch_size=4, perturbation_q=4, step_q=1),
    dict(name="block-256", block_size=256, batch_size=4, perturbation_q=4, step_q=1),
    dict(name="block-128", block_size=128, batch_size=4, perturbation_q=4, step_q=1),
    dict(name="block-64", block_size=64, batch_size=4, perturbation_q=4, step_q=1),
]

spsa_runs = {}
spsa_summaries = []

for spec in primary_sweep:
    q, frame, summary = run_two_sided(
        generations=SPSA_GENERATIONS,
        **spec,
    )
    spsa_runs[spec["name"]] = (q, frame)
    spsa_summaries.append(summary)

primary_summary_df = pd.DataFrame(spsa_summaries).sort_values(
    ["best_token_accuracy", "final_token_accuracy"], ascending=False
)
primary_summary_df

In [ ]:
plt.figure(figsize=(10, 5))
for name, (_, frame) in spsa_runs.items():
    plt.plot(frame["generation"], frame["token_accuracy"], label=name)
plt.xlabel("generation")
plt.ylabel("probe token accuracy")
plt.title("Global vs blockwise two-sided Q8.8 search")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "spsa-block-sweep.png")
plt.show()

## Focused follow-up around the best block size

The first sweep changes only block size. The next sweep takes the best non-global block and tests
whether a larger batch and/or larger integer update step materially improve signal quality.

In [ ]:
non_global = primary_summary_df[primary_summary_df["name"] != "production-global"]
best_row = non_global.iloc[0]
BEST_BLOCK = int(best_row["block_size"])
print("best block from primary sweep:", BEST_BLOCK)

focused_specs = [
    dict(name=f"block-{BEST_BLOCK}-b16", block_size=BEST_BLOCK, batch_size=16, perturbation_q=4, step_q=1),
    dict(name=f"block-{BEST_BLOCK}-b32", block_size=BEST_BLOCK, batch_size=32, perturbation_q=4, step_q=1),
    dict(name=f"block-{BEST_BLOCK}-b16-s2", block_size=BEST_BLOCK, batch_size=16, perturbation_q=4, step_q=2),
]

focused_summaries = []
for spec in focused_specs:
    q, frame, summary = run_two_sided(
        generations=SPSA_GENERATIONS,
        **spec,
    )
    spsa_runs[spec["name"]] = (q, frame)
    focused_summaries.append(summary)

focused_summary_df = pd.DataFrame(focused_summaries).sort_values(
    ["best_token_accuracy", "final_token_accuracy"], ascending=False
)
focused_summary_df

## Final decision

The report separates the architecture/numerics question from the optimizer question.

A PASS here is intentionally conservative: the notebook should not recommend a production optimizer
merely because it moves loss slightly. We want a clear improvement over `production-global`, and we
want Q8.8-aware gradient training to demonstrate that the representational path itself is viable.

In [ ]:
all_spsa = pd.concat([primary_summary_df, focused_summary_df], ignore_index=True)
production = all_spsa[all_spsa["name"] == "production-global"].iloc[0]
best_native = all_spsa.sort_values(
    ["best_token_accuracy", "final_token_accuracy"], ascending=False
).iloc[0]

fp32_final = fp32_summary["final_token_accuracy"]
q88_final = qat_ce_summary["final_token_accuracy"]
margin_final = qat_margin_summary["final_token_accuracy"]

q88_gate = q88_final >= 0.90 * max(fp32_final, 1e-9)
objective_gate = margin_final >= 0.80
native_improvement = (
    best_native["best_token_accuracy"] >= production["best_token_accuracy"] + 0.05
)
native_useful = best_native["best_token_accuracy"] >= 0.25

decision = {
    "format": "minicells.trainability.v1",
    "experiment": "MINI Cells Experiment 003 — Native Trainability Bridge",
    "status": "PASS" if (q88_gate and objective_gate and native_improvement and native_useful) else "NEEDS_ITERATION",
    "gates": {
        "q88_trainable": bool(q88_gate),
        "margin_objective_trainable": bool(objective_gate),
        "native_optimizer_materially_better_than_global": bool(native_improvement),
        "native_optimizer_reaches_useful_signal": bool(native_useful),
    },
    "gradient": {
        "fp32_final_token_accuracy": fp32_final,
        "q88_final_token_accuracy": q88_final,
        "q88_margin_final_token_accuracy": margin_final,
    },
    "production_global": production.to_dict(),
    "best_native_candidate": best_native.to_dict(),
    "fixed_integer_samples": fixed_samples,
    "run": {
        "seed": SEED,
        "full_run": FULL_RUN,
        "adam_steps": ADAM_STEPS,
        "spsa_generations": SPSA_GENERATIONS,
        "validation_examples": VAL_EXAMPLES,
        "device": str(DEVICE),
    },
}

(OUT / "decision.json").write_text(json.dumps(decision, indent=2))
print(json.dumps(decision, indent=2))

In [ ]:
# Compact artifact table for Kaggle output.
artifact_rows = []
for path in sorted(OUT.iterdir()):
    if path.is_file():
        artifact_rows.append({"file": path.name, "bytes": path.stat().st_size})
pd.DataFrame(artifact_rows)

### Interpretation

- If **Q8.8 trainable = false**, stop optimizer work and revisit quantization/forward arithmetic.
- If Q8.8 passes but **margin objective = false**, redesign the native objective before changing SPSA.
- If both pass and blockwise SPSA clearly beats global SPSA, implement the smallest winning change
  in the Rust research harness first, then require native ↔ PVM parity again.
- Do **not** move to Transport or persistent Memory until a deterministic native optimizer has a
  repeatable learning curve across multiple seeds.